# Séance 10 — Accès aux données et automatisation avancée
### Cours "Visualisation de Données"

---

> **🎯 Objectifs de la séance**
> - Comprendre la différence entre CSV, API REST et MCP comme sources de données
> - Faire des appels HTTP avec `requests` et visualiser les résultats avec Plotly
> - Encapsuler une API REST comme tool d'un agent LangGraph
> - Créer un serveur MCP minimal avec **FastMCP** et le connecter à un agent
> - Construire un pipeline complet : langage naturel → agent → API/MCP → réponse structurée

---

## Prérequis techniques

```bash
# Installation des librairies Python
pip install langchain langgraph langchain-openai langchain-mcp-adapters mcp requests pandas plotly

# Ollama doit être installé et lancé localement
# Téléchargement : https://ollama.com
# Lancement du service : ollama serve
# Téléchargement du modèle : ollama pull llama3.2
```

**Prérequis :** Séances 8 et 9 — LangGraph, agents ReAct, tools  
**Dataset utilisé :** `job_postings.csv` — 25 114 offres d'emploi, 17 colonnes

---
# PARTIE 1 — COURS THÉORIQUE
---

## Comparaison des sources de données

| Critère | CSV / fichier local | Base de données | API REST | MCP |
|---------|--------------------|-----------------|---------|----|
| **Accès** | Lecture disque | Requête SQL | HTTP GET/POST | Protocole JSON-RPC via stdio/SSE |
| **Temps réel** | Non | Dépend | Oui (souvent) | Oui |
| **Auth requise** | Non | Parfois | Souvent | Non (local) |
| **Scalabilité** | Faible | Haute | Haute | Variable |
| **Intégration LLM** | Manuel | Manuel | Via tool | Natif (conçu pour LLM) |
| **Exemples** | `job_postings.csv` | PostgreSQL | Open-Meteo, GitHub | Serveurs FastMCP locaux |

**Conclusion :** CSV = simple et rapide pour l'analyse. API = données fraîches du monde réel. MCP = pont standardisé entre outils locaux/distants et agents LLM.


## Anatomie d'une requête API REST

```
GET https://api.open-meteo.com/v1/forecast
        ?latitude=48.42
        &longitude=-71.06
        &hourly=temperature_2m
        &forecast_days=1
```

| Composant | Rôle | Exemple |
|-----------|------|---------|
| **URL de base** | Identifie le service | `https://api.open-meteo.com/v1/` |
| **Endpoint** | Ressource demandée | `/forecast` |
| **Query params** | Filtres/options | `latitude=48.42&forecast_days=1` |
| **Headers** | Méta-données (auth, format) | `Authorization: Bearer <token>` |
| **Méthode HTTP** | Type d'opération | GET (lire), POST (créer), PUT (modifier) |
| **Réponse JSON** | Données structurées | `{"hourly": {"temperature_2m": [...]}}` |

### Codes de statut HTTP courants
- `200 OK` — succès
- `400 Bad Request` — paramètres invalides
- `401 Unauthorized` — clé API manquante ou invalide
- `404 Not Found` — ressource inexistante
- `429 Too Many Requests` — quota dépassé
- `500 Internal Server Error` — erreur côté serveur

> **Open-Meteo** est une API météo *sans clé API*, idéale pour apprendre.


## MCP — Model Context Protocol

### Qu'est-ce que MCP ?

MCP (Model Context Protocol) est un **protocole ouvert** créé par Anthropic (2024) qui standardise la façon dont les LLM interagissent avec des outils externes, des ressources et des services.

### Différence API REST vs MCP

| | API REST | MCP |
|--|----------|-----|
| **Conçu pour** | Humains + machines | LLM en priorité |
| **Découverte** | Documentation manuelle | Auto-découverte (`list_tools`) |
| **Transport** | HTTP | stdio, SSE, HTTP |
| **Schema** | OpenAPI (optionnel) | JSON Schema natif |
| **Contexte** | Stateless | Peut être stateful |

### Architecture MCP

```
┌─────────────────┐     JSON-RPC      ┌──────────────────┐
│   Client MCP    │◄─────────────────►│   Serveur MCP    │
│  (agent LLM)   │   stdio / SSE      │  (FastMCP / SDK) │
└─────────────────┘                   └──────────────────┘
         │                                     │
    LangGraph                          @mcp.tool()
    create_react_agent                 fonctions Python
```

### Pourquoi MCP plutôt qu'une API classique pour les LLM ?
1. **Découverte automatique** : le LLM peut lister les outils disponibles
2. **Schémas typés** : chaque tool expose son interface (types, descriptions)
3. **Protocole unifié** : un seul client pour tous les serveurs MCP
4. **Isolation** : le serveur tourne dans son propre processus


## Quand utiliser quoi ?

| Situation | Solution recommandée |
|-----------|---------------------|
| Analyse ponctuelle de données statiques | **CSV + Pandas** |
| Données qui changent souvent (météo, cours de bourse) | **API REST** |
| Requêtes complexes sur grandes données structurées | **Base de données** |
| Exposer des fonctions Python à un LLM local | **Serveur MCP (FastMCP)** |
| Agent qui doit combiner plusieurs sources | **LangGraph + tools (API et/ou MCP)** |
| Prototypage rapide | **CSV ou API publique sans clé** |

> Dans cette séance : on commence par l'API REST (Partie 1) puis on construit des serveurs MCP (Partie 2), avant de tout combiner dans l'exercice final.


In [ ]:
# ─── SETUP — Installation et imports ───────────────────────────────────────
import subprocess, sys

# Installer langchain-mcp-adapters si absent
# Paquets requis
for pkg, import_name in [
    ('langchain-mcp-adapters', 'langchain_mcp_adapters'),
    ('langchain-openai',       'langchain_openai'),
]:
    try:
        __import__(import_name)
        print(f'✓ {pkg} déjà installé')
    except ImportError:
        print(f'Installation de {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print(f'✓ {pkg} installé')

# Imports principaux
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*LangGraphDeprecated.*')
import requests
import pandas as pd
import plotly.express as px
import os, json, webbrowser, asyncio
import nest_asyncio; nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

print('✓ Tous les imports OK')
print(f'Répertoire de travail : {os.getcwd()}')



---
# PARTIE 2 — TRAVAUX PRATIQUES
---

## Partie 1 — API REST

### 1A. Premier appel API — Open Meteo

[Open-Meteo](https://open-meteo.com/) est une API météo **gratuite et sans clé API**.
Elle retourne des prévisions horaires au format JSON.

**Endpoint utilisé :**
```
GET https://api.open-meteo.com/v1/forecast
    ?latitude=48.42
    &longitude=-71.06
    &hourly=temperature_2m
    &forecast_days=1
```

La réponse JSON contient une clé `hourly` avec des listes de timestamps et de valeurs.


In [ ]:
# NE PAS MODIFIER — 1A : premier appel API — Open Meteo (Saguenay)
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 48.42,
    'longitude': -71.06,
    'hourly': 'temperature_2m',
    'forecast_days': 1
}

response = requests.get(url, params=params)
print(f'Statut HTTP : {response.status_code}')

if response.status_code == 200:
    data = response.json()
    print(f'Clés de la réponse : {list(data.keys())}')
    temps = data['hourly']['temperature_2m']
    heures = data['hourly']['time']
    print(f'Nombre de valeurs : {len(temps)}')
    print('Premières valeurs :')
    for h, t in zip(heures[:3], temps[:3]):
        print(f'  {h} → {t}°C')
else:
    print(f'Erreur : {response.text}')


### Exercice 1A — Changer de ville

**Objectif :** Répéter l'appel API pour **Montréal** (latitude=45.51, longitude=-73.55).

Vous devez :
1. Modifier les coordonnées dans `params`
2. Faire l'appel avec `requests.get()`
3. Afficher le statut et les 3 premières températures

> Conseil : la structure du code est identique à la cellule précédente, seuls les paramètres changent.


In [ ]:
# EXERCICE 1A — Appel API pour Montréal
url = 'https://api.open-meteo.com/v1/forecast'

# TODO: Définir les paramètres pour Montréal (lat=45.51, lon=-73.55)
params = {
    'latitude': None,   # TODO: latitude de Montréal
    'longitude': None,  # TODO: longitude de Montréal
    'hourly': 'temperature_2m',
    'forecast_days': 1
}

# TODO: Faire l'appel API avec requests.get()
response = None  # TODO

# TODO: Afficher le statut HTTP
# print(f'Statut HTTP : {response.status_code}')

# TODO: Afficher les 3 premières températures
# (même pattern que la cellule 1A)


### 1B. JSON → DataFrame → graphique Plotly

Une fois les données récupérées en JSON, on les transforme en **DataFrame Pandas** pour pouvoir les visualiser facilement avec Plotly Express.

**Pipeline :**
```
API JSON → dict Python → DataFrame → px.line() → HTML
```

Le graphique est sauvegardé en HTML et ouvert dans le navigateur.


In [ ]:
# NE PAS MODIFIER — 1B : JSON → DataFrame → graphique Plotly
url = 'https://api.open-meteo.com/v1/forecast'
params = {
    'latitude': 48.42,
    'longitude': -71.06,
    'hourly': 'temperature_2m',
    'forecast_days': 1
}

response = requests.get(url, params=params)
data = response.json()

# Construire le DataFrame
df = pd.DataFrame({
    'heure': pd.to_datetime(data['hourly']['time']),
    'temperature_C': data['hourly']['temperature_2m']
})

print(df.head())
print(f'Shape : {df.shape}')

# Graphique Plotly
fig = px.line(
    df, x='heure', y='temperature_C',
    title='Températures à Saguenay — prochaines 24h',
    labels={'heure': 'Heure', 'temperature_C': 'Température (°C)'}
)
fig.update_traces(line_color='#e74c3c')
fig.update_layout(template='plotly_white')

out = 'chart_1B_temperature.html'
fig.write_html(out)
print(f'✓ Graphique sauvegardé : {out}')
webbrowser.open(f'file://{os.path.abspath(out)}')


### Exercice 1B — Ajouter la probabilité de précipitations

**Objectif :** Étendre le graphique 1B en ajoutant la variable `precipitation_probability` (en %).

Vous devez :
1. Ajouter `'precipitation_probability'` dans `params['hourly']` (séparer par virgule)
2. Créer une colonne `precip_pct` dans le DataFrame
3. Utiliser `px.line` avec `y=['temperature_C', 'precip_pct']` pour afficher les deux courbes

> Astuce : Pour `hourly`, passez une chaîne avec virgule : `'temperature_2m,precipitation_probability'`


In [ ]:
# EXERCICE 1B — Température + probabilité de précipitations
url = 'https://api.open-meteo.com/v1/forecast'

# TODO: Ajouter precipitation_probability aux paramètres
params = {
    'latitude': 48.42,
    'longitude': -71.06,
    'hourly': 'temperature_2m',  # TODO: ajouter ',precipitation_probability'
    'forecast_days': 1
}

response = requests.get(url, params=params)
data = response.json()

# TODO: Créer le DataFrame avec les deux colonnes
df = pd.DataFrame({
    'heure': pd.to_datetime(data['hourly']['time']),
    'temperature_C': data['hourly']['temperature_2m'],
    # TODO: ajouter colonne 'precip_pct' depuis data['hourly']['precipitation_probability']
})

# TODO: Créer le graphique avec les deux courbes
# fig = px.line(df, x='heure', y=['temperature_C', 'precip_pct'], title='...')

# TODO: Sauvegarder et ouvrir
out = 'chart_1B_exo.html'
# fig.write_html(out)
# webbrowser.open(f'file://{os.path.abspath(out)}')


### 1C. API comme tool d'agent LangGraph

**Idée clé :** En encapsulant un appel API dans un `@tool` LangGraph, on permet à l'agent LLM de décider **quand** et **comment** appeler l'API en fonction de la question de l'utilisateur.

```
Question (NL) → Agent → @tool get_weather() → API REST → Réponse NL
```

Le décorateur `@tool` génère automatiquement le schéma JSON que le LLM utilise pour choisir et appeler l'outil.

**Avantages :**
- L'agent peut combiner plusieurs outils
- La logique métier reste dans le tool (pas dans le prompt)
- Le LLM gère l'analyse de la réponse et sa reformulation


In [ ]:
# NE PAS MODIFIER — Setup LLM (Ollama local)
llm = ChatOpenAI(
    model='llama3.2',
    base_url='http://localhost:11434/v1',
    api_key='ollama',
    temperature=0
)

# Fonction utilitaire pour afficher les étapes de l'agent
def show_steps(result):
    messages = result.get('messages', [])
    print(f'\n' + '='*60)
    print(f'Nombre de messages : {len(messages)}')
    for i, msg in enumerate(messages):
        t = type(msg).__name__
        if hasattr(msg, 'content') and msg.content:
            content = str(msg.content)[:200]
            print(f'  [{i}] {t}: {content}')
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{i}] → Tool call: {tc["name"]}({tc.get("args", {})})')
    print('='*60 + '\n')
    final = messages[-1].content if messages else 'Aucune réponse'
    print(f'Réponse finale :\n{final}')

print('✓ LLM configuré et show_steps() défini')


async def show_mcp_steps(agent, question: str):
    """Affiche les etapes d un agent MCP en streaming."""
    print(f"► Question : {question}")
    print("─" * 60)
    async for chunk in agent.astream({"messages": [("user", question)]}):
        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Tool : {tc['name']}({tc['args']})")
            else:
                content = msg.content
                if content:
                    print(f"  ✓ {content[:200]}")
        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat : {str(obs)[:150]}")
    print("─" * 60)
print("✓ show_mcp_steps() prêt")


In [ ]:
# NE PAS MODIFIER — 1C : @tool wrapping Open Meteo
@tool
def get_weather(city: str, latitude: float = 48.42, longitude: float = -71.06) -> str:
    '''Recupere la meteo actuelle (temperature) pour une ville donnee via Open-Meteo.
    Args:
        city: Nom de la ville (pour l affichage)
        latitude: Latitude geographique de la ville
        longitude: Longitude geographique de la ville
    Returns:
        Resume textuel de la meteo actuelle
    '''
    url = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude': latitude,
        'longitude': longitude,
        'hourly': 'temperature_2m,precipitation_probability',
        'forecast_days': 1
    }
    resp = requests.get(url, params=params, timeout=10)

    if resp.status_code != 200:
        return f'Erreur API pour {city} : statut {resp.status_code}'

    data = resp.json()
    temps = data['hourly']['temperature_2m']
    precip = data['hourly']['precipitation_probability']

    temp_actuelle = temps[0]
    temp_max = max(temps)
    temp_min = min(temps)
    precip_max = max(precip) if precip else 0

    return (
        f'Meteo a {city} : '
        f'temperature actuelle {temp_actuelle}C, '
        f'min {temp_min}C / max {temp_max}C aujourd hui, '
        f'probabilite de precipitations max {precip_max}%.'
    )

print('✓ Tool get_weather() défini')
print(f'  Description : {get_weather.description[:80]}...')


In [ ]:
# NE PAS MODIFIER — 1C : agent LangGraph avec tool API

agent = create_react_agent(llm, tools=[get_weather])

result = agent.invoke({
    'messages': [('user', 'Quel temps fait-il a Saguenay ? (latitude=48.42, longitude=-71.06)')]
})

show_steps(result)


### Exercice 1C — Autre ville

**Objectif :** Utiliser le même agent pour demander la météo d'une autre ville.

Vous devez :
1. Formuler une question en français mentionnant **Montréal** avec ses coordonnées
2. Appeler `agent.invoke()` avec cette question
3. Afficher les étapes avec `show_steps()`

> Observer comment l'agent extrait les coordonnées de votre question et les passe au tool.


In [ ]:
# EXERCICE 1C — Demander la météo d'une autre ville à l'agent

# TODO: Formuler une question sur Montréal (lat=45.51, lon=-73.55)
question = None  # TODO: 'Quel temps fait-il a Montreal ? (latitude=..., longitude=...)'

# TODO: Invoquer l'agent avec cette question
# result = agent.invoke({'messages': [('user', question)]})

# TODO: Afficher les étapes
# show_steps(result)


## Partie 2 — MCP (Model Context Protocol)

### Architecture MCP : serveur / client

Un serveur MCP est un **processus Python indépendant** qui expose des fonctions (tools) via le protocole MCP. Le client MCP (ici LangGraph) se connecte à ce serveur, récupère la liste des tools disponibles, et peut les appeler.

**Flux de communication :**
```
Notebook (client)
    │
    ├─ MultiServerMCPClient
    │       │
    │       └─► subprocess: python mcp_weather_server.py
    │               │
    │               └─ FastMCP server (stdio)
    │                       ├─ tool: get_weather(city, lat, lon)
    │                       └─ tool: ...
    │
    └─ get_tools() → liste de LangChain tools → create_react_agent
```

### Transport stdio
Le transport **stdio** (standard input/output) est le plus simple : le client lance le serveur comme sous-processus et communique via stdin/stdout en JSON-RPC. Pas de port réseau à gérer.

### FastMCP
`FastMCP` est la bibliothèque Python qui simplifie la création de serveurs MCP :
- Décorateur `@mcp.tool()` pour exposer une fonction
- Typage automatique → schéma JSON pour le LLM
- Démarrage avec `mcp.run()`


### Anatomie d'un serveur FastMCP

```python
from mcp.server.fastmcp import FastMCP

# 1. Créer le serveur avec un nom descriptif
mcp = FastMCP('MonServeur')

# 2. Exposer des fonctions avec @mcp.tool()
@mcp.tool()
def ma_fonction(parametre: str) -> str:
    'Description de la fonction (utilisée par le LLM pour choisir le tool).'
    # Logique Python normale
    return f'Résultat pour {parametre}'

@mcp.tool()
def autre_fonction(n: int) -> list:
    'Retourne une liste de n éléments.'
    return list(range(n))

# 3. Point d'entrée
if __name__ == '__main__':
    mcp.run()  # Démarre le serveur MCP en mode stdio
```

**Points clés :**
- La **docstring** de chaque tool est cruciale : c'est ce que le LLM lit pour décider quel tool utiliser
- Les **types Python** génèrent automatiquement le schéma JSON (str, int, float, list, dict)
- Le serveur tourne en arrière-plan, géré par `MultiServerMCPClient`


### 2B. Créer un serveur MCP minimal

Nous allons écrire un fichier `mcp_weather_server.py` sur le disque depuis le notebook, puis le tester en nous y connectant via `MultiServerMCPClient`.

**Pourquoi écrire le fichier depuis Python ?**
Cela permet de garder tout le code dans le notebook et d'éviter de gérer un fichier externe séparément. En production, le fichier serait versioned dans le dépôt.


In [ ]:
# NE PAS MODIFIER — 2B : écrire le serveur MCP météo sur disque

server_code = "from mcp.server.fastmcp import FastMCP\nimport requests\n\nmcp = FastMCP('WeatherServer')\n\n@mcp.tool()\ndef get_weather(city: str, latitude: float = 48.42, longitude: float = -71.06) -> str:\n    'Recupere la meteo actuelle pour une ville (temperature et precipitations).'\n    url = 'https://api.open-meteo.com/v1/forecast'\n    params = {\n        'latitude': latitude,\n        'longitude': longitude,\n        'hourly': 'temperature_2m,precipitation_probability',\n        'forecast_days': 1,\n    }\n    resp = requests.get(url, params=params, timeout=10)\n    if resp.status_code != 200:\n        return f'Erreur API pour {city}: {resp.status_code}'\n    data = resp.json()\n    temps = data['hourly']['temperature_2m']\n    precip = data['hourly']['precipitation_probability']\n    temp_actuelle = temps[0]\n    temp_max = max(temps)\n    temp_min = min(temps)\n    precip_max = max(precip) if precip else 0\n    return (\n        f'Meteo a {city}: temperature actuelle {temp_actuelle}C, '\n        f'min {temp_min}C / max {temp_max}C, '\n        f'precipitations max {precip_max}%.'\n    )\n\nif __name__ == '__main__':\n    mcp.run()\n"

with open('mcp_weather_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('\u2713 mcp_weather_server.py ecrit dans', os.path.abspath('mcp_weather_server.py'))


In [ ]:
# NE PAS MODIFIER — 2B : tester le serveur MCP météo

from langchain_mcp_adapters.client import MultiServerMCPClient

async def test_mcp_tools():
    client = MultiServerMCPClient({
        'weather': {
            'command': 'python',
            'args': [os.path.abspath('mcp_weather_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools MCP disponibles ({len(tools)}) :')
    for t in tools:
        print(f'  - {t.name}: {t.description[:80]}...')
    return tools

await test_mcp_tools()


### Exercice 2B — Ajouter un tool au serveur

**Objectif :** Enrichir le serveur `mcp_weather_server.py` avec un deuxième tool : `get_forecast_7days(city, latitude, longitude)` qui retourne les températures sur **7 jours** au lieu de 1.

Vous devez :
1. Réécrire la variable `server_code` avec le nouveau tool ajouté
2. Réécrire le fichier `mcp_weather_server.py`
3. Vous reconnecter via `MultiServerMCPClient` et vérifier que les **2 tools** apparaissent

> Indice : changez `forecast_days=1` en `forecast_days=7` et calculez la température moyenne journalière.


In [ ]:
# EXERCICE 2B — Ajouter get_forecast_7days au serveur MCP

# TODO: Copier le server_code de la cellule 2B et ajouter un second @mcp.tool()
# Le nouveau tool get_forecast_7days(city, latitude, longitude) doit :
# - Appeler Open-Meteo avec forecast_days=7
# - Retourner un résumé des températures sur 7 jours

server_code_v2 = None  # TODO: construire la chaîne du serveur avec 2 tools

# TODO: Écrire le fichier
# with open('mcp_weather_server.py', 'w', encoding='utf-8') as f:
#     f.write(server_code_v2)

# TODO: Tester avec MultiServerMCPClient et afficher les 2 tools disponibles
# (copier le pattern de la cellule 2B)


### 2C. MCP + LangGraph : agent en langage naturel

On combine maintenant le serveur MCP avec un agent LangGraph.
L'agent reçoit les tools MCP via `get_tools()`, exactement comme des LangChain tools classiques.

**Pipeline complet :**
```
Question NL → create_react_agent(llm, mcp_tools) → appel MCP tool → réponse NL
```

La beauté de cette approche : **le code de l'agent ne change pas** selon que les tools viennent d'une API directe ou d'un serveur MCP. La couche d'abstraction est totale.


In [ ]:
# NE PAS MODIFIER — 2C : agent LangGraph + MCP

async def run_mcp_agent(question: str):
    client = MultiServerMCPClient({
        'weather': {
            'command': 'python',
            'args': [os.path.abspath('mcp_weather_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools charges depuis MCP : {[t.name for t in tools]}')
    agent = create_react_agent(llm, tools)
    await show_mcp_steps(agent, question)

await run_mcp_agent(
    'Quel temps fait-il a Saguenay au Quebec ? Utilise latitude=48.42 et longitude=-71.06.'
)


### Exercice 2C — Autre question

**Objectif :** Poser une question comparative qui oblige l'agent à appeler le tool **deux fois**.

Vous devez :
1. Modifier la question pour comparer la météo de deux villes
2. Observer comment l'agent appelle le tool deux fois de suite
3. Vérifier que la réponse finale compare bien les deux villes

> Question suggérée : *"Fait-il plus chaud à Montréal (lat=45.51, lon=-73.55) ou à Saguenay (lat=48.42, lon=-71.06) ?"*


In [ ]:
# EXERCICE 2C — Question comparative avec l'agent MCP

# TODO: Formuler une question qui compare Montréal et Saguenay
question = None  # TODO

# TODO: Appeler await run_mcp_agent(question)
# Observez que l'agent appellera le tool get_weather deux fois !


### 2D. Serveur MCP complet pour job_postings.csv

Nous allons maintenant créer un serveur MCP pour exposer les données du fichier `job_postings.csv` (Séance 9) via des tools analytiques.

**Tools exposés :**
1. `describe_jobs()` — aperçu général du dataset (shape, colonnes)
2. `top_industries(n)` — les N industries qui recrutent le plus
3. `salary_stats()` — statistiques sur les salaires (Minimum Pay)

**Chemin du fichier :** `../Seance9/job_postings.csv` (relatif au serveur)

> Ce pattern est très puissant : vous pouvez exposer n'importe quelle analyse Pandas comme tool MCP, et l'agent peut ensuite répondre à des questions en langage naturel sur vos données.


In [ ]:
# NE PAS MODIFIER — 2D : écrire mcp_jobs_server.py

server_code = "from mcp.server.fastmcp import FastMCP\nimport pandas as pd\nimport os\n\nmcp = FastMCP('JobsServer')\nCSV_PATH = os.path.join(os.path.dirname(__file__), '../Seance9/job_postings.csv')\n\ndef load_jobs():\n    return pd.read_csv(CSV_PATH)\n\n@mcp.tool()\ndef describe_jobs() -> str:\n    'Retourne un apercu general du dataset job_postings (nombre de lignes, colonnes).'\n    df = load_jobs()\n    cols = ', '.join(df.columns.tolist())\n    return (\n        f'Dataset job_postings: {df.shape[0]} offres d emploi, {df.shape[1]} colonnes.\\n'\n        f'Colonnes: {cols}'\n    )\n\n@mcp.tool()\ndef top_industries(n: int = 5) -> str:\n    'Retourne les N industries qui publient le plus d offres d emploi.'\n    df = load_jobs()\n    col = 'Company Industry'\n    if col not in df.columns:\n        return f'Colonne {col} introuvable. Colonnes: {list(df.columns)}'\n    top = df[col].value_counts().head(n)\n    result = f'Top {n} industries (offres d emploi):\\n'\n    for industry, count in top.items():\n        result += f'  - {industry}: {count} offres\\n'\n    return result.strip()\n\n@mcp.tool()\ndef salary_stats() -> str:\n    'Retourne des statistiques sur les salaires minimaux (colonne Minimum Pay).'\n    df = load_jobs()\n    col = 'Minimum Pay'\n    if col not in df.columns:\n        return f'Colonne {col} introuvable.'\n    s = df[col].dropna()\n    if len(s) == 0:\n        return 'Aucune donnee salariale disponible.'\n    return (\n        f'Statistiques Minimum Pay ({len(s)} offres avec salaire):\\n'\n        f'  Mediane: {s.median():.0f}\\n'\n        f'  Moyenne: {s.mean():.0f}\\n'\n        f'  Min: {s.min():.0f}\\n'\n        f'  Max: {s.max():.0f}\\n'\n        f'  Percentile 25%: {s.quantile(0.25):.0f}\\n'\n        f'  Percentile 75%: {s.quantile(0.75):.0f}'\n    )\n\nif __name__ == '__main__':\n    mcp.run()\n"

with open('mcp_jobs_server.py', 'w', encoding='utf-8') as f:
    f.write(server_code)

print('\u2713 mcp_jobs_server.py ecrit dans', os.path.abspath('mcp_jobs_server.py'))


In [ ]:
# NE PAS MODIFIER — 2D : agent LangGraph + MCP jobs

async def run_jobs_agent(question: str):
    client = MultiServerMCPClient({
        'jobs': {
            'command': 'python',
            'args': [os.path.abspath('mcp_jobs_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools charges depuis MCP Jobs : {[t.name for t in tools]}')
    agent = create_react_agent(llm, tools)
    await show_mcp_steps(agent, question)

await run_jobs_agent(
    'Quelles sont les 5 industries qui recrutent le plus ? '
    'Et quelles sont les statistiques sur les salaires minimaux ?'
)


### Exercice 2D — Étendre le serveur jobs

**Objectif :** Ajouter un quatrième tool `job_position_types()` au serveur MCP jobs qui liste les types de postes disponibles et leur fréquence.

La colonne à utiliser est `'Job Position Type'`.

Vous devez :
1. Réécrire `jobs_server_code` avec le nouveau tool
2. Sauvegarder `mcp_jobs_server.py`
3. Tester avec un agent en posant : *"Quels types de postes sont les plus recherchés ?"*

> Comparez la structure à `top_industries()` — c'est presque identique, seul le nom de colonne change.


In [ ]:
# EXERCICE 2D — Ajouter job_position_types() au serveur MCP jobs

# TODO: Réécrire jobs_server_code avec le 4ème tool :
# @mcp.tool()
# def job_position_types() -> str:
#     'Liste les types de postes et leur frequence dans job_postings.'
#     df = load_jobs()
#     col = 'Job Position Type'
#     ...

jobs_server_code_v2 = None  # TODO: construire la chaîne du serveur avec 4 tools

# TODO: Écrire le fichier
# with open('mcp_jobs_server.py', 'w', encoding='utf-8') as f:
#     f.write(jobs_server_code_v2)

# TODO: Tester
# await run_jobs_agent('Quels types de postes sont les plus recherches ?')


## Projet final — NL → API + MCP → analyse complète

### Objectif

Construire un **agent polyvalent** capable de répondre à des questions qui combinent :
- Des données **météo en temps réel** (via le serveur MCP weather)
- Des données sur les **offres d'emploi** (via le serveur MCP jobs)

**Exemple de question combinée :**
> *"Est-ce que l'industrie technologique recrute beaucoup au Canada ? Et quel temps fait-il à Saguenay en ce moment ?"*

### Ce que vous devez faire

1. Créer un agent LangGraph qui a accès **aux deux serveurs MCP** (weather + jobs) simultanément
2. Utiliser `MultiServerMCPClient` avec deux entrées dans le dictionnaire
3. Récupérer tous les tools (`get_tools()`) et les passer à `create_react_agent`
4. Poser une question combinée qui nécessite d'utiliser les deux types de tools
5. Afficher les étapes pour voir comment l'agent coordonne les appels

> Conseil : `MultiServerMCPClient` accepte plusieurs serveurs dans le même dictionnaire. Les tools de tous les serveurs sont fusionnés dans `get_tools()`.


In [ ]:
# EXERCICE FINAL — Agent combiné : MCP météo + MCP jobs

async def run_combined_agent(question: str):
    # TODO: Configurer MultiServerMCPClient avec DEUX serveurs :
    # - 'weather': mcp_weather_server.py
    # - 'jobs': mcp_jobs_server.py
    client = MultiServerMCPClient({
        # TODO: Ajouter les deux serveurs ici
    })

        # TODO: Récupérer les tools et créer l'agent
        # TODO: Récupérer les tools et créer l'agent
        tools = None  # TODO: await client.get_tools()
        print(f'Tools disponibles : {[t.name for t in tools] if tools else "TODO"}')

        agent = None  # TODO: create_react_agent(llm, tools)
        result = None  # TODO: await agent.ainvoke({'messages': [('user', question)]})

        # TODO: show_steps(result)

question_combinee = (
    'Quelles sont les industries qui recrutent le plus dans le dataset ? '
    'Et quel temps fait-il a Saguenay (latitude=48.42, longitude=-71.06) en ce moment ? '
    'Donne-moi un resume combine.'
)

# TODO: await run_combined_agent(question_combinee)


## Synthèse — Ce que vous avez appris

| Concept | Ce qu'on a vu | Pourquoi c'est utile |
|---------|--------------|---------------------|
| **API REST** | `requests.get()`, paramètres, JSON | Données en temps réel sans base de données |
| **JSON → DataFrame** | `pd.DataFrame()` depuis dict | Visualisation Plotly des données API |
| **@tool LangGraph** | Encapsuler une API comme tool | L'agent décide quand appeler l'API |
| **FastMCP** | `@mcp.tool()`, `mcp.run()` | Exposer du Python local au LLM |
| **MultiServerMCPClient** | Connexion stdio, `get_tools()` | Connecter plusieurs serveurs MCP |
| **Agent combiné** | Tools API + MCP ensemble | Répondre à des questions multi-sources |

### Prochaines étapes

- **Authentification API** : gérer les clés API avec des variables d'environnement
- **Serveurs MCP distants** : transport SSE pour des serveurs sur réseau
- **Mémoire d'agent** : persister l'état entre les conversations
- **Visualisations dans les réponses** : retourner des URLs de graphiques depuis les tools
- **Tests** : écrire des tests unitaires pour les tools MCP

> **Ressources :**
> - [Open-Meteo Docs](https://open-meteo.com/en/docs)
> - [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
> - [LangGraph Docs](https://langchain-ai.github.io/langgraph/)
> - [FastMCP Docs](https://gofastmcp.com/)
